# MSCI-tvillingar: kurser från LSEG
Kör i CodeBook, cell för cell med **Shift + Enter**. Det tar cirka 5 minuter.

När det är klart: ladda ner `msci_tvilling_priser.csv` och `msci_lseg_logg.txt` och ladda upp dem till Claude.

In [ ]:
import pandas as pd, time
try:
    import lseg.data as ld
except Exception:
    import refinitiv.data as ld
ld.open_session()
print('session öppen')

In [ ]:
RICS = ["BONAVa.ST", "BONAVb.ST", "COREa.ST", "COREb.ST", "ELUXa.ST", "ELUXb.ST", "HOLMa.ST", "HOLMb.ST", "HUFVa.ST", "HUFVc.ST^A20", "HUSQa.ST", "HUSQb.ST", "IFSa.ST^J16", "IFSb.ST^J16", "INDUa.ST", "INDUc.ST", "INVEa.ST", "INVEb.ST", "KINVa.ST", "KINVb.ST", "KLOVa.ST^G21", "KLOVb.ST^G21", "MTGa.ST", "MTGb.ST", "NCCa.ST", "NCCb.ST", "RATOa.ST", "RATOb.ST", "SAGAa.ST", "SAGAb.ST", "SCAa.ST", "SCAb.ST", "SSABa.ST", "SSABb.ST", "TWWsdba.ST^K14", "TWWsdbb.ST^K14", "VICPA.ST^F19", "VICPb.ST^F19", "VOLVa.ST", "VOLVb.ST"]
# extra stavningar att prova om en RIC saknar data
ALT = {'HOLMa.ST': ['HOLMa.ST', 'HOLMA.ST'], 'MTGa.ST': ['MTGa.ST', 'MTGA.ST', 'MTGa.ST^A26'], 'ELUXa.ST': ['ELUXa.ST', 'ELUXA.ST'], 'RATOa.ST': ['RATOa.ST'], 'NCCa.ST': ['NCCa.ST'], 'BONAVa.ST': ['BONAVa.ST']}
print(len(RICS), 'RIC')

In [ ]:
FALT = ['TR.PriceClose', 'TR.Volume', 'TR.TotalReturn1D']
rader, logg = [], []
def hamta(r):
    for forsok in range(3):
        try:
            h = ld.get_history(universe=r, fields=FALT, start='2006-01-01', end='2026-10-31', interval='daily')
            if h is not None and len(h) > 20:
                return h
            return None
        except Exception as e:
            logg.append(f'{r} försök {forsok + 1}: FEL {str(e)[:80]}'); time.sleep(3)
    return None
for r in RICS:
    h, anvand = None, r
    for kand in ALT.get(r, [r]):
        h = hamta(kand)
        if h is not None:
            anvand = kand; break
    if h is None:
        logg.append(f'{r}: inga data'); print(logg[-1]); continue
    h = h.reset_index(); h.columns = ['Date'] + [str(c) for c in h.columns[1:]]
    h['ric'] = r; h['ric_hamtad'] = anvand; rader.append(h)
    logg.append(f'{r} ({anvand}): {len(h)} dagar, {h.Date.iloc[0]} till {h.Date.iloc[-1]}'); print(logg[-1])
P = pd.concat(rader, ignore_index=True)
P.to_csv('msci_tvilling_priser.csv', index=False)
open('msci_lseg_logg.txt', 'w').write('\n'.join(logg))
print('Klart:', P.ric.nunique(), 'av', len(RICS), 'RIC. Ladda ner msci_tvilling_priser.csv och msci_lseg_logg.txt.')